# Earnings Momentum Strategy Demonstration

This notebook recreates the workflow described in `docs/earnings_momentum_strategy.md` and mirrors the MCP tool registered in `server/main.py`. We will:
- Pull a configurable basket of symbols from Yahoo Finance
- Engineer volume surge + bullish close signals
- Simulate the capped-position, fixed-hold earnings drift described in the MCP server
- Visualize recent signals and per-symbol performance

## Step 1: Parameters and Dependencies

We keep the defaults aligned with the MCP tool (20-day volume window, 2x spike threshold, 5-day hold, 3 max positions). Feel free to tweak them and rerun all cells.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_columns', None)

symbols = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]
period = "6mo"
volume_window = 20
volume_multiplier = 2.0
hold_days = 5
max_positions = 3

print(f"Universe: {', '.join(symbols)} | Period: {period}")
print(f"Volume window: {volume_window} | Spike threshold: {volume_multiplier}x")
print(f"Hold days: {hold_days} | Max positions: {max_positions}")

Universe: AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA | Period: 6mo
Volume window: 20 | Spike threshold: 2.0x
Hold days: 5 | Max positions: 3


## Step 2: Download and Normalize Daily OHLCV

We follow the server tool by downloading each symbol separately, keeping only the standard OHLCV columns, tagging each row with its ticker, and concatenating everything into one tidy frame.

In [2]:
def fetch_symbol_history(symbol: str, period: str) -> pd.DataFrame:
    """Mirror the MCP helper that pulls 1d candles and tags each row."""
    data = yf.download(
        symbol,
        period=period,
        interval="1d",
        progress=False,
        multi_level_index=False,
    )
    if data is None or data.empty:
        raise ValueError(f"No data for {symbol}")
    frame = data[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
    frame = frame.rename(columns={"Date": "date"})
    frame["symbol"] = symbol
    return frame

frames = []
failed_symbols = []
for ticker in symbols:
    try:
        frames.append(fetch_symbol_history(ticker, period))
    except Exception as exc:
        failed_symbols.append((ticker, str(exc)))

if not frames:
    raise RuntimeError("No price data was retrieved; check ticker universe or network access.")

history = pd.concat(frames, ignore_index=True)
history["date"] = pd.to_datetime(history["date"])
history.sort_values(["date", "symbol"], inplace=True)
history.reset_index(drop=True, inplace=True)

print(f"Fetched {len(history):,} rows across {history['symbol'].nunique()} symbols")
if failed_symbols:
    print("Failed downloads:")
    for item in failed_symbols:
        print(" -", item)

history.head()

C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(


Fetched 896 rows across 7 symbols


C:\Users\User\AppData\Local\Temp\ipykernel_75000\2728970914.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(


,date,Open,High,Low,Close,Volume,symbol
0,2025-07-28,213.580334,214.398618,212.612370,213.600296,37858000,AAPL
1,2025-07-28,233.350006,234.289993,232.250000,232.789993,26300100,AMZN
2,2025-07-28,193.350472,193.749863,190.544821,192.282135,38139500,GOOGL
3,2025-07-28,714.135210,723.660984,711.618942,716.561584,8715700,META
4,2025-07-28,512.277511,513.194268,508.331374,510.703033,14308000,MSFT


## Step 3: Engineer Volume and Signal Features

We compute the rolling average volume, derive the volume ratio, and tag bullish closes exactly like the MCP tool. These features feed both the signal filter and the simulator.

In [3]:
feature_frame = history.copy()
feature_frame["volume_sma"] = feature_frame.groupby("symbol")["Volume"].transform(
    lambda s: s.rolling(volume_window).mean()
)
feature_frame["volume_ratio"] = feature_frame["Volume"] / feature_frame["volume_sma"]
feature_frame["bullish_close"] = feature_frame["Close"] > feature_frame["Open"]
feature_frame = feature_frame.dropna(subset=["volume_sma", "volume_ratio"])

feature_frame[["symbol", "date", "Close", "Volume", "volume_sma", "volume_ratio", "bullish_close"]].tail()

,symbol,date,Close,Volume,volume_sma,volume_ratio,bullish_close
891,GOOGL,2026-01-28,334.750000,2640278,2.798378e+07,0.094350,False
892,META,2026-01-28,674.240784,683099,1.365537e+07,0.050024,False
893,MSFT,2026-01-28,480.190002,1361556,2.453066e+07,0.055504,False
894,NVDA,2026-01-28,191.785004,11483782,1.510608e+08,0.076021,True
895,TSLA,2026-01-28,432.878998,2071159,5.830382e+07,0.035524,True


## Step 4: Detect Volume Spike + Bullish Close Signals

Signals require both a bullish close and a volume ratio that clears the configured multiplier. We will reuse the filtered rows later when we simulate capped positions.

In [4]:
raw_signals = feature_frame[
    (feature_frame["bullish_close"]) &
    (feature_frame["volume_ratio"] >= volume_multiplier)
].copy()

print(f"Signals identified: {len(raw_signals)} (before applying position caps)")
raw_signals[["date", "symbol", "Close", "Volume", "volume_ratio"]].head()

Signals identified: 6 (before applying position caps)


,date,symbol,Close,Volume,volume_ratio
184,2025-09-03,GOOGL,230.303253,103336100,3.082598
237,2025-09-12,TSLA,395.940002,168156400,2.099005
266,2025-09-19,AAPL,245.262238,163741300,2.936752
270,2025-09-19,MSFT,516.962402,52474100,2.415985
714,2025-12-19,AAPL,273.670013,144632000,2.979547


## Step 5: Simulate Fixed-Hold Positions with Capacity Constraints

The MCP implementation enters at the close, limits concurrent exposure, and force-closes trades after `hold_days`. We translate that logic verbatim here so notebook results line up with the server tool.

In [5]:
from typing import Dict, List, Tuple


def simulate_positions(
    data: pd.DataFrame,
    hold_days: int,
    max_positions: int,
    volume_multiplier: float,
) -> Tuple[List[dict], List[dict]]:
    """Carbon-copy the server logic so notebook + MCP stay in sync."""
    active_positions: Dict[str, Dict] = {}
    trades: List[dict] = []
    signals: List[dict] = []

    grouped = data.groupby("date", sort=True)
    for trade_date, day_frame in grouped:
        records = day_frame.to_dict("records")
        record_map = {rec["symbol"]: rec for rec in records}

        # Update open positions before looking for new entries
        for symbol in list(active_positions.keys()):
            if symbol not in record_map:
                continue  # skip dates missing this ticker
            pos = active_positions[symbol]
            pos["bars_held"] += 1
            if pos["bars_held"] >= hold_days:
                exit_row = record_map[symbol]
                exit_price = float(exit_row["Close"])
                ret_pct = (exit_price - pos["entry_price"]) / pos["entry_price"] * 100
                trades.append({
                    "symbol": symbol,
                    "entry_date": pos["entry_date"].strftime("%Y-%m-%d"),
                    "exit_date": trade_date.strftime("%Y-%m-%d"),
                    "entry_price": round(pos["entry_price"], 4),
                    "exit_price": round(exit_price, 4),
                    "return_pct": round(ret_pct, 2),
                    "volume_ratio": round(pos["volume_ratio"], 2),
                    "hold_days": pos["bars_held"],
                })
                del active_positions[symbol]

        available_slots = max_positions - len(active_positions)
        if available_slots <= 0:
            continue

        candidates = [
            rec for rec in records
            if rec.get("bullish_close")
            and rec.get("symbol") not in active_positions
            and np.isfinite(rec.get("volume_ratio", np.nan))
            and rec.get("volume_ratio", 0) >= volume_multiplier
        ]
        candidates.sort(key=lambda rec: rec["volume_ratio"], reverse=True)

        for rec in candidates[:available_slots]:
            symbol = rec["symbol"]
            entry_price = float(rec["Close"])
            active_positions[symbol] = {
                "entry_date": rec["date"],
                "entry_price": entry_price,
                "volume_ratio": float(rec["volume_ratio"]),
                "bars_held": 0,
            }
            signals.append({
                "date": rec["date"].strftime("%Y-%m-%d"),
                "symbol": symbol,
                "volume_ratio": round(float(rec["volume_ratio"]), 2),
                "close": round(entry_price, 2),
            })

    if active_positions:
        for symbol, pos in list(active_positions.items()):
            symbol_history = (
                data[data["symbol"] == symbol]
                .sort_values("date")
                .dropna(subset=["Close"])
            )
            if symbol_history.empty:
                continue
            last_row = symbol_history.iloc[-1]
            exit_price = float(last_row["Close"])
            ret_pct = (exit_price - pos["entry_price"]) / pos["entry_price"] * 100
            trades.append({
                "symbol": symbol,
                "entry_date": pos["entry_date"].strftime("%Y-%m-%d"),
                "exit_date": last_row["date"].strftime("%Y-%m-%d"),
                "entry_price": round(pos["entry_price"], 4),
                "exit_price": round(exit_price, 4),
                "return_pct": round(ret_pct, 2),
                "volume_ratio": round(pos["volume_ratio"], 2),
                "hold_days": hold_days,
                "forced_exit": True,
            })
            del active_positions[symbol]

    return trades, signals


trades, signals = simulate_positions(
    feature_frame,
    hold_days=hold_days,
    max_positions=max_positions,
    volume_multiplier=volume_multiplier,
)

trades_df = pd.DataFrame(trades)
signals_df = pd.DataFrame(signals)

print(f"Trades executed: {len(trades_df)} | Recent signals stored: {len(signals_df)}")
trades_df.head()

Trades executed: 6 | Recent signals stored: 6


,symbol,entry_date,exit_date,entry_price,exit_price,return_pct,volume_ratio,hold_days
0,GOOGL,2025-09-03,2025-09-10,230.3033,239.0137,3.78,3.08,5
1,TSLA,2025-09-12,2025-09-19,395.9400,426.0700,7.61,2.10,5
2,AAPL,2025-09-19,2025-09-26,245.2622,255.2126,4.06,2.94,5
3,MSFT,2025-09-19,2025-09-26,516.9624,510.5045,-1.25,2.42,5
4,AAPL,2025-12-19,2025-12-29,273.6700,273.7600,0.03,2.98,5


## Step 6: Aggregate Metrics and Leaderboard

With the simulated trades we can reproduce the same aggregate stats the MCP tool surfaces: total trades, win rate, average return, hold duration, and per-symbol breakdowns.

In [6]:
if trades_df.empty:
    aggregate = {
        "total_trades": 0,
        "win_rate_pct": 0.0,
        "avg_return_pct": 0.0,
        "median_return_pct": 0.0,
        "avg_hold_days": float(hold_days),
    }
    per_symbol = pd.DataFrame(columns=["trades", "win_rate_pct", "avg_return_pct", "median_return_pct", "best_trade_pct", "worst_trade_pct"])
else:
    aggregate = {
        "total_trades": len(trades_df),
        "win_rate_pct": round((trades_df["return_pct"] > 0).mean() * 100, 1),
        "avg_return_pct": round(trades_df["return_pct"].mean(), 2),
        "median_return_pct": round(trades_df["return_pct"].median(), 2),
        "avg_hold_days": round(trades_df["hold_days"].mean(), 1),
    }
    per_symbol = trades_df.groupby("symbol")["return_pct"].agg(
        trades="count",
        win_rate_pct=lambda s: round((s > 0).mean() * 100, 1),
        avg_return_pct=lambda s: round(s.mean(), 2),
        median_return_pct=lambda s: round(s.median(), 2),
        best_trade_pct=lambda s: round(s.max(), 2),
        worst_trade_pct=lambda s: round(s.min(), 2),
    )

aggregate, per_symbol

({'total_trades': 6,
  'win_rate_pct': np.float64(83.3),
  'avg_return_pct': np.float64(2.72),
  'median_return_pct': np.float64(2.93),
  'avg_hold_days': np.float64(5.0)},
         trades  win_rate_pct  avg_return_pct  median_return_pct  \
 symbol                                                            
 AAPL         2         100.0            2.04               2.04   
 AMZN         1         100.0            2.08               2.08   
 GOOGL        1         100.0            3.78               3.78   
 MSFT         1           0.0           -1.25              -1.25   
 TSLA         1         100.0            7.61               7.61   
 
         best_trade_pct  worst_trade_pct  
 symbol                                   
 AAPL              4.06             0.03  
 AMZN              2.08             2.08  
 GOOGL             3.78             3.78  
 MSFT             -1.25            -1.25  
 TSLA              7.61             7.61  )

## Step 7: Visualize Signals and Performance

Plotly figures make it easy to inspect recent spikes (volume ratio vs date) alongside per-symbol hit rates. Charts are saved under `graphs/` for later comparison with other strategy notebooks.

In [7]:
graphs_dir = Path("graphs")
graphs_dir.mkdir(exist_ok=True)

if signals_df.empty:
    print("No qualifying signals were generated; skipping scatter plot.")
else:
    signals_df["date"] = pd.to_datetime(signals_df["date"])
    leaderboard = per_symbol.reset_index() if not per_symbol.empty else pd.DataFrame()

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "Recent Volume Spikes",
        "Per-Symbol Avg Return"
    ))

    fig.add_trace(
        go.Scatter(
            x=signals_df["date"],
            y=signals_df["volume_ratio"],
            mode="markers",
            marker=dict(size=10, color=signals_df["close"], colorscale="Viridis", showscale=True),
            text=[f"{sym} @ ${px:.2f}" for sym, px in zip(signals_df["symbol"], signals_df["close"])],
            name="Volume ratio",
        ),
        row=1,
        col=1,
    )

    if not leaderboard.empty:
        fig.add_trace(
            go.Bar(
                x=leaderboard["symbol"],
                y=leaderboard["avg_return_pct"],
                text=leaderboard["win_rate_pct"].apply(lambda v: f"{v:.1f}% win"),
                name="Avg return",
            ),
            row=1,
            col=2,
        )
    else:
        fig.add_trace(
            go.Bar(x=["n/a"], y=[0], name="Avg return"),
            row=1,
            col=2,
        )

    fig.update_layout(height=500, title=f"Earnings Momentum Signals ({period})")
    output_path = graphs_dir / "earnings_momentum_signals.html"
    fig.write_html(output_path)
    fig.show()
    print(f"Saved interactive plot to {output_path}")

Saved interactive plot to graphs\earnings_momentum_signals.html


## Step 8: Inspect Trade Log and Recent Entries

The MCP response exposes both capped trade history and the most recent signals for UI components. We replicate that here for manual inspection.

In [8]:
if trades_df.empty:
    print("No trades executed for the chosen parameters.")
else:
    display(trades_df.tail(10))

if signals_df.empty:
    print("No signals recorded under the current thresholds.")
else:
    display(signals_df.tail(10))

,symbol,entry_date,exit_date,entry_price,exit_price,return_pct,volume_ratio,hold_days
0,GOOGL,2025-09-03,2025-09-10,230.3033,239.0137,3.78,3.08,5
1,TSLA,2025-09-12,2025-09-19,395.9400,426.0700,7.61,2.10,5
2,AAPL,2025-09-19,2025-09-26,245.2622,255.2126,4.06,2.94,5
3,MSFT,2025-09-19,2025-09-26,516.9624,510.5045,-1.25,2.42,5
4,AAPL,2025-12-19,2025-12-29,273.6700,273.7600,0.03,2.98,5
5,AMZN,2025-12-19,2025-12-29,227.3500,232.0700,2.08,2.00,5


,date,symbol,volume_ratio,close
0,2025-09-03,GOOGL,3.08,230.30
1,2025-09-12,TSLA,2.10,395.94
2,2025-09-19,AAPL,2.94,245.26
3,2025-09-19,MSFT,2.42,516.96
4,2025-12-19,AAPL,2.98,273.67
5,2025-12-19,AMZN,2.00,227.35


## Recap

- Data + features follow `server/strategies/earnings_momentum.py`, ensuring parity with the MCP server registered in `server/main.py`.
- The notebook exposes the same knobs (universe, window, multiplier, hold days, capacity) so you can test custom universes before wiring them into the Streamlit tab or FastAPI endpoint.
- Extend this template by persisting `trades_df` to `reports_Example/` or by piping `signals_df` into the multi-strategy dashboards under `docs/`.